In [1]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()

api_keys = os.getenv("FIREWORKS_API_KEY")

url = "https://api.fireworks.ai/v1/accounts/fireworks/models"

headers = {
    "Authorization": f"Bearer {api_keys}"
}

response = requests.get(url, headers = headers)
print(response.status_code)

data = response.json()

200


In [5]:
print(data)
count = 0
for model in data['models']:
    count += 1
    print(f"Model Name: {model['name']}")
print(f"Total models: {count}")

{'models': [{'baseModelDetails': {'checkpointFormat': 'HUGGINGFACE', 'defaultPrecision': 'FP16', 'huggingfaceFiles': ['.gitattributes', 'README.md', 'config.json', 'fireworks.json', 'generation_config.json', 'pytorch_model-00001-of-00003.bin', 'pytorch_model-00002-of-00003.bin', 'pytorch_model-00003-of-00003.bin', 'pytorch_model.bin.index.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json'], 'modelType': 'llama', 'moe': False, 'parameterCount': '13016192000', 'supportsFireattention': True, 'supportsMtp': False, 'tunable': True, 'worldSize': 1}, 'calibrated': False, 'cluster': '', 'contextLength': 4096, 'conversationConfig': {'style': 'alpaca', 'system': '', 'template': ''}, 'createTime': '2024-02-29T05:52:38.739683Z', 'defaultDraftModel': '', 'defaultDraftTokenCount': 0, 'defaultSamplingParams': {'temperature': 0.9, 'top_k': 50, 'top_p': 0.6}, 'deployedModelRefs': [], 'deprecationDate': None, 'description': '(chronos-13b-v2 + Nous-Hermes-Llama

In [28]:
import os

from langchain_core.messages import HumanMessage
from langchain_fireworks import ChatFireworks


llm = ChatFireworks(
    api_key=os.getenv("FIREWORKS_API_KEY"),
    model="accounts/fireworks/models/glm-5p2"
)

messages = [
    HumanMessage(content="What is the capital of France?")
]

response = llm.invoke(messages)

print(response.content)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000021A95CE91D0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000021A95CE8E10>


The capital of France is Paris.


In [23]:
print(response.content)

The capital of France is Paris.


In [33]:
import os

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_fireworks import ChatFireworks


llm = ChatFireworks(
    api_key=os.getenv("FIREWORKS_API_KEY"),
    model="accounts/fireworks/models/deepseek-v4-flash",
    temperature=0,
    max_retries=0,
    timeout=30,
)


messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Who is oreki and which anime is he from and rate that anime out of 10 and write an essay on that anime in 100 words."),
]


response = llm.invoke(messages)
print(response)
print(response.content)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000021A97428910>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000021A97428B90>


content='**Oreki Houtarou** is the protagonist of the anime **"Hyouka"** (also known as *Classic Literature Club*). I rate this anime **9 out of 10**.\n\n**Essay on Hyouka (100 words):**  \n*Hyouka* is a masterful blend of slice-of-life and mystery, centered on the energy-conserving Oreki Houtarou, whose curiosity is ignited by the inquisitive Chitanda Eru. Kyoto Animation’s stunning visuals and subtle character animation breathe life into a seemingly mundane high school setting. Each episode unravels small, everyday puzzles that reveal deeper emotional truths about the characters. The series excels in atmosphere, pacing, and thematic depth—exploring curiosity, talent, and the beauty of ordinary moments. While the slow burn may not suit everyone, its quiet brilliance and satisfying character growth make it a hidden gem. A thoughtful, visually poetic experience that lingers long after the credits roll.' additional_kwargs={'reasoning_content': 'We need to answer: Who is Oreki? Which anim

In [1]:
from langchain_fireworks import ChatFireworks
from langchain_core.tools import tool
import os
from dotenv import load_dotenv

load_dotenv()


@tool
async def get_my_wallet_balance() -> str:
    """Get the current authenticated customer's wallet balance."""
    return "Your current wallet balance is $25.00."


model = ChatFireworks(
    api_key=os.getenv("FIREWORKS_API_KEY"),
    model="accounts/fireworks/models/deepseek-v4-flash",
    timeout=30,
)

model_with_tools = model.bind_tools(
    [get_my_wallet_balance]
)

response = await model_with_tools.ainvoke(
    "Use the wallet tool to check my current balance."
)

print("CONTENT:")
print(response.content)

print("\nTOOL CALLS:")
print(response.tool_calls)

print("\nFULL MESSAGE:")
print(response)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000014C0E94ED50>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000014C0E94D950>


CONTENT:


TOOL CALLS:
[{'name': 'get_my_wallet_balance', 'args': {}, 'id': 'chatcmpl-tool-a6d2fc1f6369b627', 'type': 'tool_call'}]

FULL MESSAGE:
content='' additional_kwargs={'reasoning_content': 'The user wants to check their wallet balance. Let me use the get_my_wallet_balance tool.', 'tool_calls': [{'index': 0, 'id': 'chatcmpl-tool-a6d2fc1f6369b627', 'type': 'function', 'function': {'name': 'get_my_wallet_balance', 'arguments': '{}'}}]} response_metadata={'token_usage': {'prompt_tokens': 275, 'total_tokens': 331, 'completion_tokens': 56}, 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'fireworks', 'model_name': 'accounts/fireworks/models/deepseek-v4-flash'} id='lc_run--019ff707-17cc-7940-8320-c5284c4f16ea-0' tool_calls=[{'name': 'get_my_wallet_balance', 'args': {}, 'id': 'chatcmpl-tool-a6d2fc1f6369b627', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 275, 'output_tokens': 56, 'total_tokens': 331}


In [37]:
print("Tracing:", os.getenv("LANGCHAIN_TRACING_V2"))
print("Project:", os.getenv("LANGCHAIN_PROJECT"))
print("API key exists:", bool(os.getenv("LANGCHAIN_API_KEY")))

Tracing: true
Project: lc-2
API key exists: True
